# ImageNet-val accuracy: pretrained vs fine-tuned (epoch 10)

50,000 val images, Resize(256) + CenterCrop(224) (CordsNet's own evaluation; paper Table 1: R8 57.90% top-1).
100 stimulus steps after 100 idle steps; accuracy at every 2 ms step. Main number: last step (200 ms).

Input: `Round3/outputs/imagenet_val/<model>.npz` (`Round3/imagenet/eval_imagenet.py`, copied from Torch).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

CN = Path("~/Desktop/DynVision/CordsNet").expanduser()
models = {"pretrained": "pretrained", "control": "control",
          "multi3_w1each": "3 areas, w1 each", "multi3_w0p33each": "3 areas, w1/3 each"}
color = {"pretrained": "0.35", "control": "#2a78d6", "3 areas, w1 each": "#8e44ad", "3 areas, w1/3 each": "#eb6834"}
PAPER_TOP1 = 0.5790
r = {label: np.load(CN / f"Round3/outputs/imagenet_val/{f}.npz") for f, label in models.items()}
t_ms = r["pretrained"]["time_ms"]
print({k: len(v["label"]) for k, v in r.items()}, "images")

## 1. Last-step accuracy

SE = binomial standard error over the 50,000 images.

In [ ]:
rows = []
for k, v in r.items():
    n = len(v["label"])
    top1, top5 = v["top1_t"][-1], v["top5_t"][-1]
    rows.append({"model": k, "top-1": top1, "top-5": top5, "top-1 SE": np.sqrt(top1 * (1 - top1) / n),
                 "Δ top-1 vs pretrained": top1 - r["pretrained"]["top1_t"][-1],
                 "Δ top-5 vs pretrained": top5 - r["pretrained"]["top5_t"][-1]})
table = pd.DataFrame(rows).set_index("model")
print(f"paper (R8, Table 1): top-1 {PAPER_TOP1:.4f}")
table.round(4)

## 2. Accuracy over time

Dashed: paper's top-1 (57.90%).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharex=True)
for ax, key, name in [(axes[0], "top1_t", "top-1"), (axes[1], "top5_t", "top-5")]:
    for k, v in r.items():
        ax.plot(t_ms, v[key], color=color[k], lw=2, label=k)
    ax.set(xlabel="time after image onset (ms)", ylabel=f"{name} accuracy", title=name, xlim=(0, 200), ylim=(0, 1))
    ax.grid(axis="y", color="0.9")
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].axhline(PAPER_TOP1, color="0.5", lw=1, ls="--")
axes[0].legend(frameon=False, loc="lower right")
plt.tight_layout()
plt.show()